In [6]:
import importlib.util
import numpy as np
import torch
from pathlib import Path
from Bio.PDB import PDBParser

MODEL = "12_04052026"

_spec = importlib.util.spec_from_file_location('pocket_model', f'./models/{MODEL}.py')
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

PocketFinder                = _mod.PocketFinder
AA_TO_IDX                   = _mod.AA_TO_IDX
PHYSCHEM                    = _mod.PHYSCHEM
compute_structural_features = _mod.compute_structural_features
compute_hse                 = _mod.compute_hse

In [7]:
CHECKPOINT = f'./models/{MODEL}.pt'

state    = torch.load(CHECKPOINT, map_location='cpu')
in_dim   = state['embedding.weight'].shape[1]
hidden   = state['embedding.weight'].shape[0]
n_layers = sum(1 for k in state if k.startswith('convs.') and k.endswith('.eps'))

model = PocketFinder(in_dim=in_dim, hidden=hidden, n_layers=n_layers)
model.load_state_dict(state)
model.eval()
print(f'loaded: in_dim={in_dim}  hidden={hidden}  n_layers={n_layers}  params={sum(p.numel() for p in model.parameters())}')

loaded: in_dim=18  hidden=256  n_layers=4  params=541505


In [8]:
_pdb_parser = PDBParser(QUIET=True)


def parse_pdb_protein(path, chain_id=None):
    structure = _pdb_parser.get_structure('p', str(path))
    coords, residue_names, res_ids = [], [], []
    for residue in structure.get_residues():
        if residue.get_id()[0] != ' ':
            continue
        chain = residue.get_parent().get_id()
        if chain_id is not None and chain != chain_id:
            continue
        res_name = residue.get_resname()
        if res_name not in AA_TO_IDX or 'CA' not in residue:
            continue
        coords.append(residue['CA'].get_vector().get_array())
        residue_names.append(res_name)
        res_ids.append(f"{chain}:{res_name}{residue.get_id()[1]}")

    coords     = np.array(coords, dtype=np.float32)
    physchem   = np.array([PHYSCHEM[r] for r in residue_names], dtype=np.float32)
    structural = compute_structural_features(coords)
    hse        = compute_hse(coords)

    def norm(x):
        std = x.std(axis=0)
        std[std < 1e-8] = 1.0
        return (x - x.mean(axis=0)) / std

    h = np.concatenate([norm(physchem), norm(structural), norm(hse)], axis=1)
    return torch.tensor(coords, dtype=torch.float32), torch.tensor(h, dtype=torch.float32), res_ids

In [9]:
# PROTEIN_PATH = Path('./data/factor_x.pdb')
# PROTEIN_PATH = Path('./data/p2rank-datasets/coach420/3efvA.pdb')
# PROTEIN_PATH = Path('./data/PDBBind/4jsz/4jsz_protein.pdb')
PROTEIN_PATH = Path('./data/p2rank-datasets/coach420/2vfcA.pdb')


THRESHOLD    = 0.6

pos, h, res_ids = parse_pdb_protein(PROTEIN_PATH)
batch = torch.zeros(len(pos), dtype=torch.long)

with torch.no_grad():
    probs = model(h, pos, batch).sigmoid()

pocket = sorted(
    [(res_ids[i], probs[i].item()) for i in range(len(res_ids)) if probs[i] > THRESHOLD],
    key=lambda x: -x[1],
)

print(f"total residues : {len(res_ids)}")
print(f"pocket residues: {len(pocket)}\n")
for res_id, prob in pocket:
    print(f"  {res_id:>12s}  p={prob:.3f}")

total residues : 272
pocket residues: 33

      A:VAL193  p=0.971
      A:PHE201  p=0.970
       A:TYR68  p=0.958
      A:PHE127  p=0.955
      A:HIS200  p=0.950
      A:GLY129  p=0.922
      A:THR131  p=0.872
       A:TYR66  p=0.870
      A:ARG215  p=0.869
      A:GLY126  p=0.866
      A:HIS107  p=0.862
      A:ALA214  p=0.853
      A:THR106  p=0.848
      A:SER199  p=0.840
      A:MET130  p=0.827
      A:GLY128  p=0.823
       A:GLU69  p=0.818
       A:HIS70  p=0.813
      A:LEU186  p=0.813
      A:HIS226  p=0.804
      A:MET206  p=0.801
      A:VAL210  p=0.779
      A:VAL125  p=0.778
      A:TRP216  p=0.751
       A:HIS29  p=0.750
      A:ASP124  p=0.739
       A:CYS67  p=0.730
      A:ARG227  p=0.717
      A:THR203  p=0.682
      A:ASP213  p=0.679
      A:VAL202  p=0.658
      A:LEU205  p=0.651
      A:ASN217  p=0.636


In [10]:
chains = set(res_id.split(':')[0] for res_id, prob in pocket)
residues_per_chain = {chain: [res_id.split(':')[-1][3:] for res_id, prob in pocket if res_id.startswith(chain)] for chain in chains}

chimera_strs = [f'color /{chain}:{",".join(residues)} #D4B045 target s' for chain, residues in residues_per_chain.items()]

chimera_strs

['color /A:193,201,68,127,200,129,131,66,215,126,107,214,106,199,130,128,69,70,186,226,206,210,125,216,29,124,67,227,203,213,202,205,217 #D4B045 target s']